> **Correction.** The comparison cells and printouts below that refer to a "0.75" TF-IDF baseline (including "+x% over 0.75") are wrong and were written before the baseline was re-measured. The actual baseline on this validation split is **0.6403 macro-F1** (see the root README). The measured BERT result for this run is **0.7439 macro-F1**, shown in the training log below.

# 🤖 ESG Classification - Phase 2: BERT Model

## From Baseline to Deep Learning

**Phase 1 Recap**: TF-IDF + Logistic Regression → ~0.75 F1  
**Phase 2 Goal**: BERT Fine-tuning → **0.80-0.85 F1** (+10-15% improvement!)

### 🎓 What You'll Learn:
1. **BERT Tokenization** - How BERT processes text (like notebook 06)
2. **Transfer Learning** - Pre-trained NLP models (like VGG16 in notebook 07)
3. **Multi-label Classification** - Binary Cross Entropy loss
4. **PyTorch Training** - Custom dataset, data loader, training loop
5. **Model Fine-tuning** - Unfreezing layers for better performance

Let's build something powerful! 🚀

In [1]:
# Import libraries
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from transformers import BertTokenizer
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("   ⚠️  No GPU detected - training will be slower (CPU)")

print("\n✅ Libraries imported successfully")

🖥️  Device: cuda
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
   Memory: 4.29 GB

✅ Libraries imported successfully


## 1. Load and Prepare Data

We'll use the same preprocessing from Phase 1, but now we'll tokenize with BERT.

In [2]:
# Load data
train = pd.read_csv('../go-data-science-5-0/train.csv')
test = pd.read_csv('../go-data-science-5-0/test.csv')

# Preprocess
from src.preprocessing import preprocess_dataframe, stratified_split_multilabel

LABEL_COLS = ['E', 'S', 'G', 'non_ESG']

print("🧹 Preprocessing data...")
train_clean = preprocess_dataframe(train.copy())
test_clean = preprocess_dataframe(test.copy())

# Train/validation split (stratified for multi-label)
train_df, val_df = stratified_split_multilabel(
    train_clean,
    LABEL_COLS,
    test_size=0.2,
    random_state=42
)

print(f"\n✅ Data prepared:")
print(f"   Training:   {len(train_df):,} samples")
print(f"   Validation: {len(val_df):,} samples")
print(f"   Test:       {len(test_clean):,} samples")

🧹 Preprocessing data...
Train size: 21373, Validation size: 5324

Label distribution in train:
E           942
S          7927
G          7084
non_ESG    9110
dtype: int64

Label distribution in validation:
E           236
S          1982
G          1771
non_ESG    2278
dtype: int64

✅ Data prepared:
   Training:   21,373 samples
   Validation: 5,324 samples
   Test:       2,000 samples


## 2. BERT Tokenization

**Key Concept**: Unlike TF-IDF which counts words, BERT understands **context**.

BERT uses **WordPiece tokenization**:
- Splits words into subwords (e.g., "walking" → "walk" + "##ing")
- Adds special tokens: `[CLS]` (start) and `[SEP]` (end)
- Maximum 512 tokens per sequence

This is from **Notebook 06** in your course!

In [3]:
# Load BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print("🔤 BERT Tokenizer loaded!")
print(f"   Vocabulary size: {tokenizer.vocab_size:,}")
print(f"   Max length: 512 tokens")

# Example: See how BERT tokenizes
example_text = train_df.iloc[0]['text']

# Tokenize
tokens = tokenizer.tokenize(example_text)
token_ids = tokenizer.encode(example_text, add_special_tokens=True)

print(f"\n📝 Example text:")
print(f"   {example_text[:200]}...")
print(f"\n🔢 Tokenization:")
print(f"   Original length: {len(example_text)} characters")
print(f"   Number of tokens: {len(tokens)}")
print(f"   First 10 tokens: {tokens[:10]}")
print(f"   Token IDs: {token_ids[:10]}...")

# Decode back
decoded = tokenizer.decode(token_ids)
print(f"\n🔄 Decoded text:")
print(f"   {decoded[:200]}...")
print(f"\n   Notice: [CLS] at start, [SEP] at end!")

## 3. Create PyTorch DataLoaders

**Similar to Course**: Remember how we created data loaders for images?  
Now we do the same for text!

Our custom `ESGDataset` class:
- Takes text and tokenizes it
- Returns `input_ids`, `attention_mask`, and `labels`
- Batches data efficiently

In [4]:
from src.data_loader import create_data_loaders

# Hyperparameters
MAX_LENGTH = 256  # Shorter than 512 to speed up training
BATCH_SIZE = 16   # Reduce if GPU memory issues

print("📦 Creating data loaders...")

train_loader, val_loader, test_loader = create_data_loaders(
    train_df=train_df,
    val_df=val_df,
    test_df=test_clean,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    label_columns=LABEL_COLS
)

print(f"✅ Data loaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches: {len(val_loader)}")
print(f"   Test batches: {len(test_loader)}")

# Inspect one batch
print(f"\n🔍 Sample batch:")
sample_batch = next(iter(train_loader))
print(f"   Input IDs shape: {sample_batch['input_ids'].shape}")
print(f"   Attention mask shape: {sample_batch['attention_mask'].shape}")
print(f"   Labels shape: {sample_batch['labels'].shape}")

📦 Creating data loaders...
✅ Data loaders created:
   Train batches: 1336
   Val batches: 333
   Test batches: 125

🔍 Sample batch:
   Input IDs shape: torch.Size([16, 256])
   Attention mask shape: torch.Size([16, 256])
   Labels shape: torch.Size([16, 4])


## 4. Build BERT Classifier

**Architecture** (similar to VGG16 in course):
```
Input Text
    ↓
BERT Tokenizer (preprocessing)
    ↓
BERT Base Model (pretrained features - like VGG16 layers)
    ↓
[CLS] Token Representation (768 dimensions)
    ↓
Dropout (regularization)
    ↓
Linear Layer (768 → 4 classes)
    ↓
Sigmoid (multi-label probabilities)
```

**Transfer Learning**: BERT learned language from millions of documents.  
We just add a classifier on top and fine-tune!

In [5]:
from src.models import get_model

# Model configuration
MODEL_TYPE = 'bert'  # or 'bert-lstm' for advanced version
BERT_MODEL_NAME = 'bert-base-uncased'
DROPOUT = 0.3
FREEZE_BERT = False  # Set True for faster training (but lower accuracy)

print("🤖 Creating BERT classifier...")

model = get_model(
    model_type=MODEL_TYPE,
    n_classes=4,
    bert_model_name=BERT_MODEL_NAME,
    dropout=DROPOUT,
    freeze_bert=FREEZE_BERT,
    device=device
)

print(f"✅ Model created!")
print(f"   Type: {MODEL_TYPE}")
print(f"   Base model: {BERT_MODEL_NAME}")
print(f"   Dropout: {DROPOUT}")
print(f"   BERT frozen: {FREEZE_BERT}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\n📊 Model statistics:")
print(f"   Total parameters: {total_params:,}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Frozen parameters: {total_params - trainable_params:,}")

# Show architecture (first few layers)
print(f"\n🏗️  Model architecture:")
print(model)

🤖 Creating BERT classifier...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model created!
   Type: bert
   Base model: bert-base-uncased
   Dropout: 0.3
   BERT frozen: False

📊 Model statistics:
   Total parameters: 109,485,316
   Trainable parameters: 109,485,316
   Frozen parameters: 0

🏗️  Model architecture:
BERTClassifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
           

## 5. Training Configuration

**Key Hyperparameters**:
- **Learning Rate**: 2e-5 (BERT sweet spot)
- **Epochs**: 3-4 (BERT learns fast!)
- **Warmup**: 10% of steps (gradual LR increase)
- **Loss**: Binary Cross Entropy with Logits (multi-label)

**Why these values?**
- BERT is already trained, so small LR to avoid "catastrophic forgetting"
- Few epochs because we're fine-tuning, not training from scratch
- Warmup stabilizes training at the start

In [6]:
from src.train_utils import train_model

# Training hyperparameters
LEARNING_RATE = 2e-5  # Recommended for BERT
N_EPOCHS = 3          # Start with 3, can increase if needed

# Save path for best model
SAVE_PATH = Path('../models/bert_best_model.pt')
SAVE_PATH.parent.mkdir(exist_ok=True)

print("⚙️  Training configuration:")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Epochs: {N_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Max length: {MAX_LENGTH}")
print(f"   Device: {device}")
print(f"   Save path: {SAVE_PATH}")
print(f"\n   Estimated time: ~{N_EPOCHS * len(train_loader) * BATCH_SIZE / 1000:.1f} min on GPU")
print(f"                    ~{N_EPOCHS * len(train_loader) * BATCH_SIZE / 100:.1f} min on CPU")

⚙️  Training configuration:
   Learning rate: 2e-05
   Epochs: 3
   Batch size: 16
   Max length: 256
   Device: cuda
   Save path: ..\models\bert_best_model.pt

   Estimated time: ~64.1 min on GPU
                    ~641.3 min on CPU


## 6. Train the Model! 🚀

**This will take 5-15 minutes on GPU, 30-60 minutes on CPU.**

Watch for:
- Train/Val loss decreasing
- Val F1 score increasing
- Best model saved when F1 improves

**Compare to baseline**:
- Baseline (TF-IDF): ~0.75 F1
- Target (BERT): >0.80 F1
- If lower, we'll troubleshoot!

In [7]:
# Train the model
print("🚀 Starting BERT training...\n")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=N_EPOCHS,
    learning_rate=LEARNING_RATE,
    device=device,
    save_path=str(SAVE_PATH),
    label_names=LABEL_COLS
)

print("\n🎉 Training complete!")

🚀 Starting BERT training...

🚀 Starting training...
Device: cuda
Epochs: 3
Learning rate: 2e-05
Warmup steps: 400 / 4008


Epoch 1 [Valid]: 100%|██████████| 333/333 [18:49<00:00,  3.39s/it, loss=0.296]



📊 Epoch 1/3 Summary (5887.4s)
──────────────────────────────────────────────────────────────────────
Train Loss: 0.4098 | Train Acc: 0.7954
Val Loss:   0.3068 | Val Acc:   0.8600
Val Macro F1: 0.7085
Per-class F1: E=0.517, S=0.890, G=0.708, non_ESG=0.720
💾 Saved best model (F1: 0.7085) to ..\models\bert_best_model.pt


Epoch 2 [Valid]: 100%|██████████| 333/333 [15:46<00:00,  2.84s/it, loss=0.284] 



📊 Epoch 2/3 Summary (5416.1s)
──────────────────────────────────────────────────────────────────────
Train Loss: 0.2890 | Train Acc: 0.8690
Val Loss:   0.2892 | Val Acc:   0.8673
Val Macro F1: 0.7393
Per-class F1: E=0.621, S=0.896, G=0.719, non_ESG=0.721
💾 Saved best model (F1: 0.7393) to ..\models\bert_best_model.pt


Epoch 3 [Valid]: 100%|██████████| 333/333 [16:08<00:00,  2.91s/it, loss=0.295] 



📊 Epoch 3/3 Summary (5530.4s)
──────────────────────────────────────────────────────────────────────
Train Loss: 0.2575 | Train Acc: 0.8874
Val Loss:   0.2887 | Val Acc:   0.8720
Val Macro F1: 0.7439
Per-class F1: E=0.622, S=0.905, G=0.703, non_ESG=0.746
💾 Saved best model (F1: 0.7439) to ..\models\bert_best_model.pt

✅ Training complete!
Best validation F1: 0.7439

🎉 Training complete!


## 7. Visualize Training Progress

In [1]:
# Plot training history
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history['epochs'], history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['epochs'], history['val_loss'], label='Val Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['epochs'], history['train_acc'], label='Train Accuracy', marker='o')
axes[1].plot(history['epochs'], history['val_acc'], label='Val Accuracy', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1 Score
axes[2].plot(history['epochs'], history['val_f1'], label='Val Macro F1', marker='o', color='green', linewidth=2)
axes[2].axhline(y=0.75, color='red', linestyle='--', label='Baseline (TF-IDF)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('F1 Score')
axes[2].set_title('Validation F1 Score', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n📈 Best Validation F1: {max(history['val_f1']):.4f}")
print(f"   Improvement over baseline: +{(max(history['val_f1']) - 0.75)*100:.1f}%")

NameError: name 'plt' is not defined

## 8. Load Best Model & Final Evaluation

In [ ]:
# Load best model
print("📥 Loading best model...")

checkpoint = torch.load(SAVE_PATH)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✅ Loaded model from epoch {checkpoint['epoch']}")
print(f"   Best Val F1: {checkpoint['val_f1']:.4f}")

# Final evaluation on validation set
from src.train_utils import predict
from src.evaluation import calculate_metrics, print_metrics_report

print("\n🔮 Final evaluation on validation set...")

val_preds, val_probs = predict(model, val_loader, device)
val_labels = val_df[LABEL_COLS].values

# Calculate comprehensive metrics
metrics = calculate_metrics(
    val_labels,
    val_preds,
    val_probs,
    LABEL_COLS
)

# Print detailed report
print_metrics_report(metrics, LABEL_COLS)

## 9. Comparison: Baseline vs BERT

Let's see how much we improved!

In [ ]:
# Comparison table
comparison = {
    'Metric': ['Macro F1', 'E F1', 'S F1', 'G F1', 'non_ESG F1', 'Training Time'],
    'Baseline (TF-IDF)': ['~0.75', '~0.45', '~0.85', '~0.80', '~0.85', '<1 min'],
    'BERT (Phase 2)': [
        f"{metrics['macro_f1']:.4f}",
        f"{metrics['E_f1']:.4f}",
        f"{metrics['S_f1']:.4f}",
        f"{metrics['G_f1']:.4f}",
        f"{metrics['non_ESG_f1']:.4f}",
        f"~{N_EPOCHS * 5} min"
    ]
}

comparison_df = pd.DataFrame(comparison)

print("\n" + "="*70)
print("📊 BASELINE vs BERT COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# Calculate improvements
baseline_f1 = 0.75  # From Phase 1
improvement = ((metrics['macro_f1'] - baseline_f1) / baseline_f1) * 100

print(f"\n🎯 Overall Improvement: +{improvement:.1f}%")
print(f"   {'🎉 GREAT!' if improvement > 5 else '😐 Could be better...'}")

if improvement < 5:
    print("\n💡 Tips to improve:")
    print("   - Train for more epochs (4-5)")
    print("   - Try unfreezing BERT (set FREEZE_BERT=False)")
    print("   - Increase MAX_LENGTH to 512")
    print("   - Use 'bert-large' instead of 'bert-base'")

## 10. Generate Kaggle Submission

In [ ]:
# Predict on test set
print("🔮 Generating predictions for test set...")

test_preds, test_probs = predict(model, test_loader, device)

# Create submission file
submission = pd.DataFrame({
    'id': test_clean['id'],
    'E': test_preds[:, 0].astype(int),
    'S': test_preds[:, 1].astype(int),
    'G': test_preds[:, 2].astype(int),
    'non_ESG': test_preds[:, 3].astype(int)
})

# Save
submission_path = '../submissions/bert_submission.csv'
submission.to_csv(submission_path, index=False)

print(f"✅ Submission saved to: {submission_path}")
print(f"\nSubmission preview:")
print(submission.head(10))

# Check distribution
print(f"\n📊 Prediction distribution:")
for col in LABEL_COLS:
    count = submission[col].sum()
    pct = (count / len(submission)) * 100
    print(f"   {col:10s}: {count:4d} / {len(submission)} ({pct:5.2f}%)")

print("\n🚀 Ready to submit to Kaggle!")

## 11. Key Learnings & Next Steps

### 🎓 What We Learned:
1. **BERT Tokenization**: WordPiece, special tokens, attention masks
2. **Transfer Learning**: Using pre-trained models for new tasks
3. **Multi-label Classification**: Binary Cross Entropy, sigmoid activation
4. **PyTorch Training**: Custom datasets, data loaders, training loops
5. **Fine-tuning Strategy**: Learning rate, warmup, gradient clipping

### 📊 Results:
- **Baseline (TF-IDF)**: ~0.75 F1
- **BERT**: {:.4f} F1
- **Improvement**: +{:.1f}%

### 🚀 Phase 3: Advanced Optimizations (Next!)

**What's coming**:
1. **Class Imbalance**: Focal loss, weighted BCE for rare 'E' class
2. **Long Text Handling**: Chunking, hierarchical models for 512+ tokens
3. **Data Augmentation**: Back-translation, synonym replacement
4. **Model Variants**: Try RoBERTa, DeBERTa (better than BERT!)
5. **Advanced Training**: Mixed precision (FP16), gradient accumulation

**Target**: 0.85-0.90 F1 → Top 10 position! 🏆

### 💾 Save Your Work:
```bash
git add .
git commit -m "[Phase 2] BERT model complete - F1 = X.XX"
git push
```

In [ ]:
# Print final summary
print("\n" + "="*70)
print("🎉 PHASE 2 COMPLETE!")
print("="*70)
print(f"Model Type: BERT-base-uncased")
print(f"Best Val F1: {max(history['val_f1']):.4f}")
print(f"Baseline F1: 0.75")
print(f"Improvement: +{((max(history['val_f1']) - 0.75) / 0.75 * 100):.1f}%")
print(f"\nSubmission File: {submission_path}")
print("\n🎯 Next: Upload to Kaggle and start Phase 3!")
print("="*70)